# 02 - Data Cleaning

Objectif : garder les colonnes utiles, convertir les nombres et exporter un fichier nettoye.

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

# Fichier source : donnees brutes a nettoyer
raw_path = project_root / 'data' / 'raw' / 'Building_Energy_Benchmarking_Data,_2015-Present_20260704.csv'

# Fichier de sortie : donnees nettoyees creees par ce notebook
cleaned_path = project_root / 'data' / 'cleaned' / 'buildings_clean.csv'

## Selectionner les colonnes utiles

In [2]:
df_raw = pd.read_csv(raw_path, low_memory=False)
raw_row_count = len(df_raw)

columns_to_keep = {
    'OSEBuildingID': 'building_id',
    'DataYear': 'data_year',
    'BuildingName': 'building_name',
    'BuildingType': 'building_type',
    'Neighborhood': 'neighborhood',
    'YearBuilt': 'year_built',
    'NumberofFloors': 'number_of_floors',
    'PropertyGFATotal': 'property_gfa_total',
    'ENERGYSTARScore': 'energy_star_score',
    'SiteEUI(kBtu/sf)': 'site_eui',
    'SiteEnergyUse(kBtu)': 'site_energy_use',
    'Electricity(kWh)': 'electricity_kwh',
    'NaturalGas(therms)': 'natural_gas_therms',
    'TotalGHGEmissions': 'total_ghg_emissions',
    'GHGEmissionsIntensity': 'ghg_emissions_intensity',
    'ComplianceStatus': 'compliance_status'
}

df = df_raw[list(columns_to_keep.keys())].rename(columns=columns_to_keep).copy()
df.head()

,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,site_energy_use,electricity_kwh,natural_gas_therms,total_ghg_emissions,ghg_emissions_intensity,compliance_status
0,1,2024,MAYFLOWER PARK HOTEL,NonResidential,DOWNTOWN,1927,12,"88,434",59.0,61.70000076,"7,113,958","1,045,040","15,986",263.3,2.98,Not Compliant
1,2,2024,PARAMOUNT HOTEL,NonResidential,DOWNTOWN,1996,11,"103,566",85.0,71.5,"6,330,664","787,838","36,426",208.6,2.36,Compliant
2,3,2024,WESTIN HOTEL (Parent Building),NonResidential,DOWNTOWN,1969,41,"956,110",71.0,81.69999695,"67,613,264","11,279,080","58,726","2,418.2",3.18,Compliant
3,5,2024,HOTEL MAX,NonResidential,DOWNTOWN,1926,10,"61,320",50.0,86,"5,273,981","796,976","11,648",190.1,3.1,Compliant
4,8,2024,WARWICK SEATTLE HOTEL,NonResidential,DOWNTOWN,1980,18,"175,580",87.0,97.09999847,"11,983,959","1,349,025","73,811",417.9,3.68,Compliant


## Nettoyer les types et les valeurs

In [3]:
numeric_columns = [
    'building_id', 'data_year', 'year_built', 'number_of_floors',
    'property_gfa_total', 'energy_star_score', 'site_eui', 'site_energy_use',
    'electricity_kwh', 'natural_gas_therms', 'total_ghg_emissions',
    'ghg_emissions_intensity'
]

numeric_updates = {}
for column in numeric_columns:
    values = df[column].astype('string').str.replace(',', '', regex=False).str.strip()
    values = values.mask(values.isin(['', 'NA', 'nan', '<NA>']))
    numeric_updates[column] = pd.to_numeric(values, errors='coerce')

df = df.assign(**numeric_updates)

text_columns = ['building_name', 'building_type', 'neighborhood', 'compliance_status']
text_updates = {}
for column in text_columns:
    text_updates[column] = df[column].fillna('Unknown').astype(str).str.strip().str.title()

df = df.assign(**text_updates)

rows_before_cleaning = len(df)
df = df.drop_duplicates().copy()

# On retire les lignes sans valeurs essentielles pour les KPI.
critical_columns = [
    'building_id', 'data_year', 'year_built', 'number_of_floors',
    'property_gfa_total', 'site_eui', 'site_energy_use', 'electricity_kwh',
    'total_ghg_emissions', 'ghg_emissions_intensity'
]
df = df.dropna(subset=critical_columns).copy()

# Regles simples de coherence metier.
df = df[df['property_gfa_total'] > 0].copy()
df = df[df['site_eui'] > 0].copy()
df = df[df['site_energy_use'] > 0].copy()
df = df[df['number_of_floors'] > 0].copy()
df = df[df['year_built'] <= df['data_year']].copy()

# ENERGY STAR et gaz naturel sont utiles, mais pas toujours renseignes.
# On garde une trace des valeurs remplacees avec des colonnes indicatrices.
df = df.assign(
    energy_star_score_was_missing=df['energy_star_score'].isna().astype(int),
    natural_gas_was_missing=df['natural_gas_therms'].isna().astype(int)
)

# Si le score ENERGY STAR manque, on remplace par la mediane pour garder la ligne.
df = df.assign(energy_star_score=df['energy_star_score'].fillna(df['energy_star_score'].median()))

# Si le gaz naturel manque, on considere 0 pour faciliter les agregations.
df = df.assign(natural_gas_therms=df['natural_gas_therms'].fillna(0))

integer_columns = ['building_id', 'data_year', 'year_built', 'number_of_floors']
df[integer_columns] = df[integer_columns].astype('int64')

df.head()

,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,site_energy_use,electricity_kwh,natural_gas_therms,total_ghg_emissions,ghg_emissions_intensity,compliance_status,energy_star_score_was_missing,natural_gas_was_missing
0,1.0,2024.0,Mayflower Park Hotel,Nonresidential,Downtown,1927.0,12.0,88434.0,59.0,61.700001,7113958.0,1045040.0,15986.0,263.3,2.98,Not Compliant,0,0
1,2.0,2024.0,Paramount Hotel,Nonresidential,Downtown,1996.0,11.0,103566.0,85.0,71.500000,6330664.0,787838.0,36426.0,208.6,2.36,Compliant,0,0
2,3.0,2024.0,Westin Hotel (Parent Building),Nonresidential,Downtown,1969.0,41.0,956110.0,71.0,81.699997,67613264.0,11279080.0,58726.0,2418.2,3.18,Compliant,0,0
3,5.0,2024.0,Hotel Max,Nonresidential,Downtown,1926.0,10.0,61320.0,50.0,86.000000,5273981.0,796976.0,11648.0,190.1,3.10,Compliant,0,0
4,8.0,2024.0,Warwick Seattle Hotel,Nonresidential,Downtown,1980.0,18.0,175580.0,87.0,97.099998,11983959.0,1349025.0,73811.0,417.9,3.68,Compliant,0,0


## Controle qualite

In [4]:
quality_check = pd.DataFrame({
    'column_name': df.columns,
    'missing_values': df.isna().sum().values,
    'missing_rate': (df.isna().mean().values * 100).round(2)
})

cleaning_summary = pd.DataFrame({
    'metric': [
        'Lignes brutes',
        'Lignes apres nettoyage',
        'Lignes retirees',
        'Part des lignes retirees (%)',
        'Scores ENERGY STAR completes',
        'Valeurs gaz naturel completees'
    ],
    'value': [
        raw_row_count,
        len(df),
        raw_row_count - len(df),
        round((raw_row_count - len(df)) / raw_row_count * 100, 2),
        int(df['energy_star_score_was_missing'].sum()),
        int(df['natural_gas_was_missing'].sum())
    ]
})

display(cleaning_summary)
quality_check

,column_name,missing_values,missing_rate
0,building_id,0,0.0
1,data_year,0,0.0
2,building_name,0,0.0
3,building_type,0,0.0
4,neighborhood,0,0.0
5,year_built,0,0.0
6,number_of_floors,0,0.0
7,property_gfa_total,0,0.0
8,energy_star_score,0,0.0
9,site_eui,0,0.0


### Conclusion du controle qualite

Le dataset final ne contient plus de valeurs manquantes. Les lignes sans valeurs essentielles pour les KPI ont ete retirees, ainsi que les cas simples incoherents : surface nulle, consommation nulle, nombre d'etages egal a 0 ou annee de construction posterieure a l'annee de reporting.

Les scores ENERGY STAR manquants ont ete remplaces par la mediane pour conserver les batiments dans l'analyse. Les valeurs manquantes de gaz naturel ont ete remplacees par 0, car cela peut correspondre a des batiments qui n'utilisent pas cette energie.

Pour garder une analyse transparente, deux colonnes indicatrices ont ete ajoutees : `energy_star_score_was_missing` et `natural_gas_was_missing`. Elles permettent d'identifier les valeurs completees pendant le nettoyage.

## Exporter les donnees nettoyees

In [5]:
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(cleaned_path, index=False)
print(f'Fichier exporte : {cleaned_path}')
print(f'Lignes exportees : {len(df):,}')

Fichier exporte : c:\Users\djami\Desktop\Projects\Energy-Performance-Dashboard\data\cleaned\buildings_clean.csv
Lignes exportees : 33,350
